In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for analyzing sales growth by therapeutic area for Q1 and Q2 2023 in Databricks
# Purpose: Compare sales performance across therapeutic areas between Q1 and Q2 2023, calculate growth percentage, and rank by top growth
# Author: Giang Nguyen
# Date: 2025-10-13
# Description: Reads from Unity Catalog table 'purgo_databricks.purgo_playground.hcp_overall_performance', extracts year/quarter, aggregates sales, pivots Q1/Q2, calculates growth percentage, and outputs sorted results

# from pyspark.sql import SparkSession  # SparkSession is already available in Databricks
from pyspark.sql import functions as F  
from pyspark.sql.types import StringType, IntegerType, LongType, DateType, DoubleType  

def validate_and_prepare_source_df(df):
    """
    Validates and prepares the source DataFrame for analysis.

    Args:
        df (pyspark.sql.DataFrame): Source DataFrame

    Returns:
        pyspark.sql.DataFrame: DataFrame with correct types and required columns
    """
    # Ensure required columns exist
    required_cols = ["Therapeutic_Area", "Sales_Amount", "Sales_Date"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")
    # Cast columns to correct types
    df = df.withColumn("Therapeutic_Area", F.col("Therapeutic_Area").cast(StringType())) \
           .withColumn("Sales_Amount", F.col("Sales_Amount").cast(LongType())) \
           .withColumn("Sales_Date", F.col("Sales_Date").cast(DateType()))
    # Filter out rows with nulls in required columns
    df = df.filter(
        F.col("Therapeutic_Area").isNotNull() &
        (F.trim(F.col("Therapeutic_Area")) != "") &
        F.col("Sales_Amount").isNotNull() &
        F.col("Sales_Date").isNotNull()
    )
    return df

def extract_year_quarter(df):
    """
    Extracts Year and Quarter from Sales_Date and filters for 2023 Q1/Q2.

    Args:
        df (pyspark.sql.DataFrame): Input DataFrame

    Returns:
        pyspark.sql.DataFrame: DataFrame with Year, Quarter columns, filtered for 2023 Q1/Q2
    """
    df = df.withColumn("Year", F.year("Sales_Date")) \
           .withColumn("Quarter", F.quarter("Sales_Date"))
    df = df.filter((F.col("Year") == 2023) & (F.col("Quarter").isin([1,2])))
    return df

def aggregate_sales(df):
    """
    Aggregates total sales by Therapeutic_Area, Year, Quarter.

    Args:
        df (pyspark.sql.DataFrame): DataFrame with Year and Quarter

    Returns:
        pyspark.sql.DataFrame: Aggregated DataFrame
    """
    return df.groupBy("Therapeutic_Area", "Year", "Quarter") \
             .agg(F.sum("Sales_Amount").alias("Total_Sales"))

def pivot_quarters(df):
    """
    Pivots the aggregated sales to have Q1_Sales and Q2_Sales columns.

    Args:
        df (pyspark.sql.DataFrame): Aggregated DataFrame

    Returns:
        pyspark.sql.DataFrame: Pivoted DataFrame
    """
    df_pivot = df.groupBy("Therapeutic_Area", "Year") \
                 .pivot("Quarter", [1,2]) \
                 .agg(F.first("Total_Sales"))
    df_pivot = df_pivot.withColumn("Q1_Sales", F.coalesce(F.col("1"), F.lit(0)).cast(LongType())) \
                       .withColumn("Q2_Sales", F.coalesce(F.col("2"), F.lit(0)).cast(LongType()))
    return df_pivot

def calculate_growth(df):
    """
    Calculates growth percentage from Q1 to Q2 sales.

    Args:
        df (pyspark.sql.DataFrame): Pivoted DataFrame

    Returns:
        pyspark.sql.DataFrame: DataFrame with Growth_Percentage column
    """
    return df.withColumn(
        "Growth_Percentage",
        F.when(F.col("Q1_Sales") != 0,
               ((F.col("Q2_Sales") - F.col("Q1_Sales")) / F.col("Q1_Sales") * F.lit(100)).cast(DoubleType())
        ).otherwise(F.lit(None).cast(DoubleType()))
    )

def select_and_sort(df):
    """
    Selects required columns and sorts by Growth_Percentage descending.

    Args:
        df (pyspark.sql.DataFrame): DataFrame with growth calculation

    Returns:
        pyspark.sql.DataFrame: Final output DataFrame
    """
    return df.select(
        "Therapeutic_Area", "Year", "Q1_Sales", "Q2_Sales", "Growth_Percentage"
    ).orderBy(F.col("Growth_Percentage").desc_nulls_last())

# Read source table
df_src = spark.table("purgo_databricks.purgo_playground.hcp_overall_performance")

# Validate and prepare source DataFrame
df_valid = validate_and_prepare_source_df(df_src)

# CTE: Extract year and quarter, filter for 2023 Q1/Q2
cte_year_quarter = extract_year_quarter(df_valid)

# CTE: Aggregate sales by area and quarter
cte_agg_sales = aggregate_sales(cte_year_quarter)

# CTE: Pivot Q1/Q2 sales
cte_pivot = pivot_quarters(cte_agg_sales)

# CTE: Calculate growth percentage
cte_growth = calculate_growth(cte_pivot)

# Final output: select and sort
df_final = select_and_sort(cte_growth)

# Display the final result (for Databricks notebook, uncomment if needed)
# df_final.show()

# spark.stop()  # Do not stop SparkSession in Databricks
